# Phase 3 — Feature Engineering
## Project: Machine Learning-based Ransomware Detection Using Low-level Memory Access Patterns
### Reference: Hirano & Kobayashi, IEEE CSR 2022

**Goal:** Transform raw memory access events into 18-dimensional feature vectors
using a sliding time window approach.

**Approach:**
- Window size  = 10 seconds (Twindow)
- Step size    = 1 second
- Trial length = 60 seconds
- Windows per trial = 50
- Total rows expected = 1970 trials × 50 windows = ~98,500 rows

**Output:** features.parquet — one row per window, ready for ML models

In [25]:
import pandas as pd
import numpy as np
import os

# paths
PROCESSED = r"C:\Users\Shiva\Downloads\RANSMAP_PROJECT\DATA\processed"
OUTPUT    = r"C:\Users\Shiva\Downloads\RANSMAP_PROJECT\DATA\processed\features.parquet"

# window parameters — from IEEE paper
WINDOW_SEC = 10   # Twindow
STEP_SEC   = 1    # slide by 1 second each time
TRIAL_SEC  = 60   # total trial duration

print("Libraries loaded!")
print(f"Window size : {WINDOW_SEC}s")
print(f"Step size   : {STEP_SEC}s")
print(f"Trial length: {TRIAL_SEC}s")
print(f"Windows per trial: {TRIAL_SEC - WINDOW_SEC + 1}")

Libraries loaded!
Window size : 10s
Step size   : 1s
Trial length: 60s
Windows per trial: 51


## 1. Load the Data

We load all 6 parquet files — one per memory/storage access type.
Only loading the columns we need for feature engineering.

In [26]:
# Don't load all files at once — process one at a time
# Define file info for looping later

FILES = {
    "mem_write":     {"path": os.path.join(PROCESSED, "mem_write_combined.parquet"),     "addr_col": "GPA"},
    "mem_read":      {"path": os.path.join(PROCESSED, "mem_read_combined.parquet"),      "addr_col": "GPA"},
    "mem_exec":      {"path": os.path.join(PROCESSED, "mem_exec_combined.parquet"),      "addr_col": "GPA"},
    "mem_readwrite": {"path": os.path.join(PROCESSED, "mem_readwrite_combined.parquet"), "addr_col": "GPA"},
    "ata_write":     {"path": os.path.join(PROCESSED, "ata_write_combined.parquet"),     "addr_col": "LBA"},
    "ata_read":      {"path": os.path.join(PROCESSED, "ata_read_combined.parquet"),      "addr_col": "LBA"},
}

print("File registry ready!")
for name, info in FILES.items():
    size_gb = os.path.getsize(info["path"]) / (1024**3)
    print(f"  {name:<20} : {size_gb:.1f} GB")

File registry ready!
  mem_write            : 4.8 GB
  mem_read             : 4.4 GB
  mem_exec             : 0.0 GB
  mem_readwrite        : 0.0 GB
  ata_write            : 13.0 GB
  ata_read             : 6.8 GB


## 2. Single Window Calculator

Before processing all 1970 trials, we test on ONE trial first.
This catches bugs cheaply before running the full loop.

### Testing on One Trial First
Before processing all 1970 trials, we test our logic on 
one WannaCry trial to catch bugs early.

In [27]:
# Load one WannaCry trial to test window logic
import pyarrow.parquet as pq

cols = ["time_sec", "GPA", "entropy", "page_type",
        "class_name", "trial_id", "label", "is_malicious"]

pf       = pq.ParquetFile(FILES["mem_write"]["path"])
n_groups = pf.metadata.num_row_groups

# Search row groups until we find WannaCry
trial_df = None
for i in range(n_groups):
    chunk = pf.read_row_group(i, columns=cols).to_pandas()
    wc    = chunk[chunk["class_name"] == "WannaCry"]
    if len(wc) > 0:
        trial_id = wc["trial_id"].iloc[0]
        trial_df = wc[wc["trial_id"] == trial_id].copy()
        print(f"Found WannaCry in row group {i}")
        break
    del chunk

print(f"Trial ID  : {trial_id}")
print(f"Shape     : {trial_df.shape}")
print(f"Time range: {trial_df['time_sec'].min()} to {trial_df['time_sec'].max()}")
print(f"Duration  : {trial_df['time_sec'].max() - trial_df['time_sec'].min()} seconds")
trial_df.head()

Found WannaCry in row group 47
Trial ID  : WannaCry-20230511_13-58-03
Shape     : (552396, 8)
Time range: 1683780843 to 1683781038
Duration  : 195 seconds


,time_sec,GPA,entropy,page_type,class_name,trial_id,label,is_malicious
0,1683780843,7.163100e+09,0.753787,2,WannaCry,WannaCry-20230511_13-58-03,malicious,1
1,1683780843,6.731505e+09,0.797222,2,WannaCry,WannaCry-20230511_13-58-03,malicious,1
2,1683780843,4.276093e+09,0.009880,2,WannaCry,WannaCry-20230511_13-58-03,malicious,1
3,1683780843,6.152877e+09,0.705952,2,WannaCry,WannaCry-20230511_13-58-03,malicious,1
4,1683780843,6.893042e+09,0.804316,2,WannaCry,WannaCry-20230511_13-58-03,malicious,1


In [28]:
# Normalize time to start from 0 — critical step!
# time_sec is Unix timestamp (e.g. 1683780843)
# We need relative time starting from 0 for our windows to work

trial_df["time_rel"] = trial_df["time_sec"] - trial_df["time_sec"].min()

print(f"Before normalization: {trial_df['time_sec'].min()} to {trial_df['time_sec'].max()}")
print(f"After normalization : {trial_df['time_rel'].min()} to {trial_df['time_rel'].max()} seconds")
print(f"\nFirst 5 relative times:")
print(trial_df["time_rel"].head())

Before normalization: 1683780843 to 1683781038
After normalization : 0 to 195 seconds

First 5 relative times:
0    0
1    0
2    0
3    0
4    0
Name: time_rel, dtype: int64


In [29]:
# Test one window — t=0 to t=10 seconds
t_start = 0
t_end   = t_start + WINDOW_SEC

window = trial_df[
    (trial_df["time_rel"] >= t_start) &
    (trial_df["time_rel"] <= t_end)
]

print(f"Window t={t_start} to t={t_end}")
print(f"Rows in window : {len(window)}")
print(f"\nFeatures from this window:")

# Shannon entropy average (H)
entropy_avg = window["entropy"].mean()

# Page type counts
count_4kb  = (window["page_type"] == 2).sum()
count_2mb  = (window["page_type"] == 3).sum()
count_mmio = (window["page_type"] == 0).sum()

# Address variance (V)
addr_var = window["GPA"].var()

print(f"  entropy_avg  : {entropy_avg:.6f}")
print(f"  count_4kb    : {count_4kb}")
print(f"  count_2mb    : {count_2mb}")
print(f"  count_mmio   : {count_mmio}")
print(f"  addr_var     : {addr_var:.2f}")

Window t=0 to t=10
Rows in window : 16022

Features from this window:
  entropy_avg  : 0.008380
  count_4kb    : 187
  count_2mb    : 0
  count_mmio   : 0
  addr_var     : 309166623732793344.00


## 3. Single Window Feature Calculator

Now we wrap the window logic into a clean calculator.
This computes all 5 features for one window of one file type.

In [30]:
def compute_window(df_window, addr_col):
    
    # entropy — only in mem_write, mem_readwrite, ata_write
    entropy_avg = df_window["entropy"].mean() if "entropy" in df_window.columns else 0.0
    
    # page_type — only in memory files, NOT ata files
    if "page_type" in df_window.columns:
        count_4kb  = (df_window["page_type"] == 2).sum()
        count_2mb  = (df_window["page_type"] == 3).sum()
        count_mmio = (df_window["page_type"] == 0).sum()
    else:
        count_4kb  = 0
        count_2mb  = 0
        count_mmio = 0

    # Normalize address before variance — fixes float32 precision loss
    # Raw GPA/LBA values are in billions — divide by 1e9 to bring to GB range
    addr_normalised = df_window[addr_col] / 1e9
    addr_var = addr_normalised.var()
    
    return {
        "entropy_avg": entropy_avg,
        "count_4kb":   count_4kb,
        "count_2mb":   count_2mb,
        "count_mmio":  count_mmio,
        "addr_var":    addr_var if not np.isnan(addr_var) else 0.0
    }

print("compute_window updated with address normalisation!")

compute_window updated with address normalisation!


## 4. Single Trial Processor

Now we slide the window across the full 60 seconds of one trial.
Each slide produces one row of 5 features.
51 windows per trial (t=0→10, t=1→11, ... t=50→60)

In [31]:
def process_trial(trial_df, addr_col, window_sec=10, step_sec=1, trial_sec=60):
    """
    Slides window across one full trial and returns list of feature dicts.
    
    Steps:
    1. Normalize time to start from 0
    2. Slide window from t=0 to t=trial_sec
    3. Compute 5 features per window
    4. Return list of feature dicts
    """
    
    # Step 1 — normalize time
    trial_df = trial_df.copy()
    trial_df["time_rel"] = trial_df["time_sec"] - trial_df["time_sec"].min()
    
    # metadata — same for all windows in this trial
    meta = {
        "trial_id":     trial_df["trial_id"].iloc[0],
        "class_name":   trial_df["class_name"].iloc[0],
        "label":        trial_df["label"].iloc[0],
        "is_malicious": trial_df["is_malicious"].iloc[0],
    }
    
    results = []
    
    # Step 2 — slide window
    for t_start in range(0, trial_sec - window_sec + 1, step_sec):
        t_end  = t_start + window_sec
        
        window = trial_df[
            (trial_df["time_rel"] >= t_start) &
            (trial_df["time_rel"] <= t_end)
        ]
        
        if len(window) == 0:
            continue
        
        # Step 3 — compute features
        features = compute_window(window, addr_col)
        features["window_start"] = t_start
        features.update(meta)
        
        results.append(features)
    
    return results

# Test on our WannaCry trial
results = process_trial(trial_df, "GPA")

print(f"Windows generated : {len(results)}")
print(f"\nFirst window features:")
for k, v in results[0].items():
    print(f"  {k:<15} : {v}")

Windows generated : 51

First window features:
  entropy_avg     : 0.008379890583455563
  count_4kb       : 187
  count_2mb       : 0
  count_mmio      : 0
  addr_var        : 0.3091666102409363
  window_start    : 0
  trial_id        : WannaCry-20230511_13-58-03
  class_name      : WannaCry
  label           : malicious
  is_malicious    : 1


In [32]:
# Convert results to DataFrame to see it clearly
trial_features = pd.DataFrame(results)

print(f"Shape : {trial_features.shape}")
print(f"\nFeature columns:")
print(trial_features.columns.tolist())
print(f"\nFirst 5 windows:")
trial_features.head()

Shape : (51, 10)

Feature columns:
['entropy_avg', 'count_4kb', 'count_2mb', 'count_mmio', 'addr_var', 'window_start', 'trial_id', 'class_name', 'label', 'is_malicious']

First 5 windows:


,entropy_avg,count_4kb,count_2mb,count_mmio,addr_var,window_start,trial_id,class_name,label,is_malicious
0,0.008380,187,0,0,0.309167,0,WannaCry-20230511_13-58-03,WannaCry,malicious,1
1,0.005309,153,0,0,0.107783,1,WannaCry-20230511_13-58-03,WannaCry,malicious,1
2,0.004375,128,0,0,0.093823,2,WannaCry-20230511_13-58-03,WannaCry,malicious,1
3,0.004115,125,0,0,0.097332,3,WannaCry-20230511_13-58-03,WannaCry,malicious,1
4,0.004364,141,0,0,0.102544,4,WannaCry-20230511_13-58-03,WannaCry,malicious,1


## 5. Full Dataset Loop

Now we scale up — process all 1970 trials across all 6 file types.
Each file type contributes 5 features → 6 × 5 = 30 features total per window.
We process one file type at a time to avoid RAM issues.
Final result: all file types merged into one feature table.

In [35]:
import pyarrow.parquet as pq
import gc

all_features = {}

for ft, info in FILES.items():
    
    # Check if already saved to disk — skip if yes
    save_path = os.path.join(PROCESSED, f"features_{ft}.parquet")
    if os.path.exists(save_path):
        all_features[ft] = pd.read_parquet(save_path)
        print(f"  {ft:<20} already saved — loaded {all_features[ft].shape}")
        continue
    
    print(f"\nProcessing {ft}...")
    
    pf       = pq.ParquetFile(info["path"])
    n_groups = pf.metadata.num_row_groups
    addr_col = info["addr_col"]
    cols     = ["time_sec", addr_col, "entropy", "page_type",
                "class_name", "trial_id", "label", "is_malicious"]
    
    # ata files don't have page_type — adjust columns
    if ft in ["ata_write", "ata_read"]:
        cols = ["time_sec", addr_col, "entropy",
                "class_name", "trial_id", "label", "is_malicious"]
    
    ft_results  = []
    trials_done = 0
    
    for i in range(0, n_groups, 5):
        batch_dfs = []
        for j in range(i, min(i + 5, n_groups)):
            chunk = pf.read_row_group(j, columns=cols).to_pandas()
            batch_dfs.append(chunk)
            del chunk
        
        batch = pd.concat(batch_dfs, ignore_index=True)
        del batch_dfs
        
        for trial_id in batch["trial_id"].unique():
            trial_df = batch[batch["trial_id"] == trial_id]
            windows  = process_trial(trial_df, addr_col)
            ft_results.extend(windows)
            trials_done += 1
        
        del batch
        gc.collect()
        
        pct = min((i + 5) / n_groups * 100, 100)
        print(f"  {pct:5.1f}%  |  {trials_done} trials done", end="\r")
    
    all_features[ft] = pd.DataFrame(ft_results)
    
    # Save immediately after each file type
    all_features[ft].to_parquet(save_path, index=False)
    print(f"\n  Done! Shape: {all_features[ft].shape} — saved!")
    
    del ft_results
    gc.collect()

print("\nAll file types processed!")

  mem_write            already saved — loaded (106431, 10)
  mem_read             already saved — loaded (105764, 10)
  mem_exec             already saved — loaded (47485, 10)
  mem_readwrite        already saved — loaded (76885, 10)

Processing ata_write...
  100.0%  |  2216 trials done
  Done! Shape: (110169, 10) — saved!

Processing ata_read...
  100.0%  |  2114 trials done
  Done! Shape: (105186, 10) — saved!

All file types processed!


In [34]:
# Save all 4 completed file types immediately
import os

for ft in ["mem_write", "mem_read", "mem_exec", "mem_readwrite"]:
    if ft in all_features:
        save_path = os.path.join(PROCESSED, f"features_{ft}.parquet")
        all_features[ft].to_parquet(save_path, index=False)
        print(f"Saved {ft} → {all_features[ft].shape}")

Saved mem_write → (106431, 10)
Saved mem_read → (105764, 10)
Saved mem_exec → (47485, 10)
Saved mem_readwrite → (76885, 10)


In [36]:
# Load all 6 saved feature files
feat_files = {
    "write":     "features_mem_write.parquet",
    "read":      "features_mem_read.parquet",
    "exec":      "features_mem_exec.parquet",
    "readwrite": "features_mem_readwrite.parquet",
    "ata_write": "features_ata_write.parquet",
    "ata_read":  "features_ata_read.parquet",
}

dfs = {}
for suffix, fname in feat_files.items():
    df = pd.read_parquet(os.path.join(PROCESSED, fname))
    # Rename feature columns with suffix
    df = df.rename(columns={
        "entropy_avg": f"entropy_avg_{suffix}",
        "count_4kb":   f"count_4kb_{suffix}",
        "count_2mb":   f"count_2mb_{suffix}",
        "count_mmio":  f"count_mmio_{suffix}",
        "addr_var":    f"addr_var_{suffix}",
    })
    dfs[suffix] = df
    print(f"Loaded {suffix}: {df.shape}")

Loaded write: (106431, 10)
Loaded read: (105764, 10)
Loaded exec: (47485, 10)
Loaded readwrite: (76885, 10)
Loaded ata_write: (110169, 10)
Loaded ata_read: (105186, 10)


In [37]:
# Merge all 6 on trial_id + window_start + metadata columns
meta_cols = ["trial_id", "window_start", "class_name", "label", "is_malicious"]

# Start with mem_write as base
features = dfs["write"][meta_cols + [c for c in dfs["write"].columns if c not in meta_cols]]

# Merge remaining 5 one by one
for suffix in ["read", "exec", "readwrite", "ata_write", "ata_read"]:
    feat_cols = [c for c in dfs[suffix].columns if c not in meta_cols]
    features = features.merge(
        dfs[suffix][meta_cols + feat_cols],
        on=meta_cols,
        how="outer"
    )

print(f"Final features shape : {features.shape}")
print(f"Columns : {features.columns.tolist()}")
print(f"\nFirst 3 rows:")
features.head(3)

Final features shape : (137406, 35)
Columns : ['trial_id', 'window_start', 'class_name', 'label', 'is_malicious', 'entropy_avg_write', 'count_4kb_write', 'count_2mb_write', 'count_mmio_write', 'addr_var_write', 'entropy_avg_read', 'count_4kb_read', 'count_2mb_read', 'count_mmio_read', 'addr_var_read', 'entropy_avg_exec', 'count_4kb_exec', 'count_2mb_exec', 'count_mmio_exec', 'addr_var_exec', 'entropy_avg_readwrite', 'count_4kb_readwrite', 'count_2mb_readwrite', 'count_mmio_readwrite', 'addr_var_readwrite', 'entropy_avg_ata_write', 'count_4kb_ata_write', 'count_2mb_ata_write', 'count_mmio_ata_write', 'addr_var_ata_write', 'entropy_avg_ata_read', 'count_4kb_ata_read', 'count_2mb_ata_read', 'count_mmio_ata_read', 'addr_var_ata_read']

First 3 rows:


,trial_id,window_start,class_name,label,is_malicious,entropy_avg_write,count_4kb_write,count_2mb_write,count_mmio_write,addr_var_write,...,entropy_avg_ata_write,count_4kb_ata_write,count_2mb_ata_write,count_mmio_ata_write,addr_var_ata_write,entropy_avg_ata_read,count_4kb_ata_read,count_2mb_ata_read,count_mmio_ata_read,addr_var_ata_read
0,AESCrypt-20220916_21-19-09,0,AESCrypt,benign,0,0.000786,101.0,0.0,0.0,0.109294,...,0.472027,0.0,0.0,0.0,0.001022,0.0,0.0,0.0,0.0,0.000645
1,AESCrypt-20220916_21-19-09,1,AESCrypt,benign,0,0.000371,58.0,0.0,0.0,1.400526,...,0.471885,0.0,0.0,0.0,0.001018,0.0,0.0,0.0,0.0,0.000670
2,AESCrypt-20220916_21-19-09,2,AESCrypt,benign,0,0.000375,60.0,0.0,0.0,1.360199,...,0.471885,0.0,0.0,0.0,0.001018,0.0,0.0,0.0,0.0,0.000694


In [38]:
# Drop entropy columns from files that don't have real entropy
# mem_read and mem_exec have entropy column but values are all zeros
# keeping them would add noise to the model

features = features.drop(columns=[
    "entropy_avg_read",
    "entropy_avg_exec"
])

print(f"Shape after dropping spurious entropy columns: {features.shape}")
print(f"Remaining columns: {features.shape[1]}")
print(f"\nFeature columns now ({features.shape[1] - 5} features):")
feat_cols = [c for c in features.columns
             if c not in ["trial_id", "window_start", "class_name", "label", "is_malicious"]]
print(feat_cols)

Shape after dropping spurious entropy columns: (137406, 33)
Remaining columns: 33

Feature columns now (28 features):
['entropy_avg_write', 'count_4kb_write', 'count_2mb_write', 'count_mmio_write', 'addr_var_write', 'count_4kb_read', 'count_2mb_read', 'count_mmio_read', 'addr_var_read', 'count_4kb_exec', 'count_2mb_exec', 'count_mmio_exec', 'addr_var_exec', 'entropy_avg_readwrite', 'count_4kb_readwrite', 'count_2mb_readwrite', 'count_mmio_readwrite', 'addr_var_readwrite', 'entropy_avg_ata_write', 'count_4kb_ata_write', 'count_2mb_ata_write', 'count_mmio_ata_write', 'addr_var_ata_write', 'entropy_avg_ata_read', 'count_4kb_ata_read', 'count_2mb_ata_read', 'count_mmio_ata_read', 'addr_var_ata_read']


In [48]:
# Sanity check — features should now separate malicious from benign
print("=== SANITY CHECK ===\n")

# Address variance distribution check
print("Address variance (write):")
print(features.groupby("class_name")["addr_var_write"]
      .mean()[["WannaCry", "Idle"]].round(2))

# 2. Does entropy differ between malicious and benign?
print("\nMean entropy_avg_write by label:")
print(features.groupby("label")["entropy_avg_write"].mean().round(6))

# 3. Missing values check
print(f"\nMissing values: {features.isnull().sum().sum()}")

# 4. Class distribution
print(f"\nRows per label:")
print(features["label"].value_counts())

=== SANITY CHECK ===

Address variance (write):
class_name
WannaCry     1.51
Idle        14.57
Name: addr_var_write, dtype: float64

Mean entropy_avg_write by label:
label
benign       0.013625
malicious    0.004542
Name: entropy_avg_write, dtype: float32

Missing values: 0

Rows per label:
label
malicious    94652
benign       42754
Name: count, dtype: int64


In [40]:
# Fix 1 — fix unknown labels (mix classes)
mix = [c for c in features["class_name"].unique()
       if "_" in c and "Conti_0" not in c]
features.loc[features["class_name"].isin(mix), "label"]        = "malicious"
features.loc[features["class_name"].isin(mix), "is_malicious"] = 1

# Fix 2 — fill missing values with 0
# Missing means that file type had no events in that window — 0 is correct
features = features.fillna(0)

# Verify fixes
print(f"Missing values after fix : {features.isnull().sum().sum()}")
print(f"\nLabel distribution:")
print(features["label"].value_counts())
print(f"\nShape : {features.shape}")

Missing values after fix : 0

Label distribution:
label
malicious    94652
benign       42754
Name: count, dtype: int64

Shape : (137406, 33)


In [41]:
# Save corrected features
save_path = os.path.join(PROCESSED, "features_final.parquet")
features.to_parquet(save_path, index=False)

mb = os.path.getsize(save_path) / (1024 * 1024)
print(f"Saved corrected features_final.parquet!")
print(f"Size     : {mb:.1f} MB")
print(f"Shape    : {features.shape}")
print(f"Features : {features.shape[1] - 5} meaningful features")
print(f"\nLabel distribution:")
print(features["label"].value_counts())
print(f"\nMissing values: {features.isnull().sum().sum()}")

Saved corrected features_final.parquet!
Size     : 6.1 MB
Shape    : (137406, 33)
Features : 28 meaningful features

Label distribution:
label
malicious    94652
benign       42754
Name: count, dtype: int64

Missing values: 0


## Conclusion

Phase 3 Feature Engineering is complete.

**What we built:**
- Sliding window approach (10s window, 1s step, 51 windows per trial)
- 5 features per window per file type
- 28 meaningful features across 6 file types
- 137,406 feature vectors from 1,970 trials

**Fixes applied:**
- Address variance normalized (divided by 1e9) to fix float32 precision loss
- Dropped entropy_avg_read and entropy_avg_exec (zero values, not real signal)
- Mix class labels fixed (unknown → malicious)

**Output:** `features_final.parquet` (6.1 MB, 28 features) — ready for ML models

**Next → Phase 4: Model Building**
- Random Forest
- SVM
- kNN
- XGBoost